```
conda create -y -n guide_reference_genome3
conda activate guide_reference_genome3
mamba install -y bioconda::r-seqinr

mamba install -y r-irkernel
mamba install -y conda-forge::r-dplyr
```

In [61]:
require(seqinr)
require(dplyr)
require(data.table)

Loading required package: data.table

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘data.table’”


In [62]:
##############Define variables and paths
barcode_length = 20

# INPUT REQUIRED
#Set directory with the genome
setwd("/Genomics/pritykinlab/share/perturbseq/fromMarc/new_data/trim28_setdb1_data/reference_genome/")

# INPUT REQUIRED
barcodes_raw <- read.csv("guide_seq.csv",
  col.names=c("gene", "guide_name", "sequence"))

######## Create fasta and gtf files
barcodes <- unique(barcodes_raw$sequence)

In [63]:
output_fasta_dir <- "/Genomics/pritykinlab/share/perturbseq/fromMarc/new_data/trim28_setdb1_data/reference_genome/per_sequence_fasta/"
combined_fasta_file <- "/Genomics/pritykinlab/share/perturbseq/fromMarc/new_data/trim28_setdb1_data/reference_genome/combined_genome.fa"

for (i in barcodes) { 
  filename <- paste(output_fasta_dir, i, ".fasta", sep = "")
  write.fasta(sequences = list(i), names = i, file.out = filename)
}
system("cat /Genomics/pritykinlab/share/perturbseq/fromMarc/new_data/trim28_setdb1_data/reference_genome/per_sequence_fasta/*.fasta > /Genomics/pritykinlab/share/perturbseq/fromMarc/new_data/trim28_setdb1_data/reference_genome/combined_genome.fa")

In [64]:
gtf_file <- "/Genomics/pritykinlab/share/perturbseq/fromMarc/new_data/trim28_setdb1_data/reference_genome/annotation_barcodes.gtf"

In [65]:
is_first_entry <- TRUE

for (i in barcodes) { 

  # GTF file creation
  chromosome_name=paste(i,"_chrom",sep="")
  custom_chromosome_length = barcode_length
  gene_id=paste(i, "_barcode_gene", sep="")
  gene_id_char=paste("gene_id",paste('"', gene_id, '"', sep=""),sep=" ")
  gene_name_char=paste("gene_name",paste('"', gene_id, '"', sep=""),sep=" ")
  
  transcript_id=paste(i, "_barcode_transcript", sep="")
  transcript_id_char=paste("transcript_id", paste('"', transcript_id, '"', sep=""),sep=" ")
  transcript_name_char=paste("transcript_name",paste('"', gene_id, '"', sep=""),sep=" ")
  
  exon_id=paste(i, "_barcode_exon", sep="")
  exon_id_char=paste("exon_id", paste('"', exon_id, '"', sep=""),sep=" ")
  
  #gft_line_gene
  gtf_line_gene = paste(
    paste(i, 
          "havana", 
          "gene",
          "1",
          custom_chromosome_length,
          ".",
          "+",
          ".",
          paste(gene_id_char, 
                'gene_version "5"', 
                gene_name_char,
                'gene_source "havana"', 
                'gene_biotype "lincRNA"',
                sep="; "), 
          sep="\t"),
    ";",sep="")
  
  #gtf_line_transcript
  gtf_line_transcript = paste(
    paste(i, 
          "havana", 
          "transcript",
          "1",
          custom_chromosome_length,
          ".",
          "+",
          ".",
          paste(gene_id_char, 
                'gene_version "5"', 
                transcript_id_char, 
                'transcript_version "2"',
                gene_name_char,
                'gene_source "havana"', 
                'gene_biotype "lincRNA"',
                transcript_name_char,
                'transcript_source "havana"',
                'transcript_biotype "lincRNA"',
                'tag "basic"',
                'transcript_support_level "NA"',
                sep="; "), 
          sep="\t"),
    ";",sep="")
  
  #gtf_line_exon
  gtf_line_exon = paste(
    paste(i, 
          "havana", 
          "exon",
          "1",
          custom_chromosome_length,
          ".",
          "+",
          ".",
          paste(gene_id_char, 
                'gene_version "5"', 
                transcript_id_char, 
                'transcript_version "2"',
                'exon_number "1"',
                gene_name_char,
                'gene_source "havana"', 
                'gene_biotype "lincRNA"',
                transcript_name_char,
                'transcript_source "havana"',
                'transcript_biotype "lincRNA"',
                exon_id_char,
                'exon_version "1"',
                'tag "basic"',
                'transcript_support_level "NA"',
                sep="; "), 
          sep="\t"),
    ";",sep="")

      # Write to the GTF file
      if (is_first_entry) {
        write(gtf_line_gene, file = gtf_file, append = FALSE)
        write(gtf_line_transcript, file = gtf_file, append = TRUE)
        write(gtf_line_exon, file = gtf_file, append = TRUE)
        is_first_entry <- FALSE
      } else {
        write(gtf_line_gene, file = gtf_file, append = TRUE)
        write(gtf_line_transcript, file = gtf_file, append = TRUE)
        write(gtf_line_exon, file = gtf_file, append = TRUE)
      }
  
  }

